In [1]:
# Clone the repository
!git clone https://github.com/kvshashankpai/MyChatbot.git
%cd MyChatbot

Cloning into 'MyChatbot'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 109 (delta 30), reused 104 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 14.64 MiB | 15.57 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/MyChatbot


In [2]:
# Install dependencies
!pip install -r requirements.txt

## Data Preparation

The processed data should already be available in the repository. Let's verify and prepare if needed.

In [3]:
# Check if processed data exists
!ls -la data/processed/

total 52864
drwxr-xr-x 2 root root     4096 Apr 14 09:23 .
drwxr-xr-x 5 root root     4096 Apr 14 09:23 ..
-rw-r--r-- 1 root root 22673620 Apr 14 09:23 corpus.txt
-rw-r--r-- 1 root root 15273300 Apr 14 09:23 empathetic_full.jsonl
-rw-r--r-- 1 root root  1542854 Apr 14 09:23 empathetic_test.jsonl
-rw-r--r-- 1 root root 12207507 Apr 14 09:23 empathetic_train.jsonl
-rw-r--r-- 1 root root  1522939 Apr 14 09:23 empathetic_val.jsonl
-rw-r--r-- 1 root root   135604 Apr 14 09:23 esconv_test.jsonl
-rw-r--r-- 1 root root   621375 Apr 14 09:23 esconv_train.jsonl
-rw-r--r-- 1 root root   127887 Apr 14 09:23 esconv_val.jsonl
-rw-r--r-- 1 root root       73 Apr 14 09:23 strategy_counts.json


In [4]:
# If data is missing, run preparation (this may take a while)
# !python scripts/prepare_data.py

## Training Phase 1

Now let's train the model on the EmpatheticDialogues dataset.

In [5]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

CUDA available: True
GPU: Tesla T4
GPU memory: 14.6 GB


In [6]:
# Train Phase 1
# Adjust batch_size and grad_accum based on GPU memory
!python train_phase1.py --batch_size 8 --grad_accum 8 --epochs 10

[Phase 1] device=cuda
[Phase 1] vocab=10000  pad_id=0
  [Dataset] Loaded 50057 samples from data/processed/empathetic_train.jsonl
  [Dataset] Loaded 6256 samples from data/processed/empathetic_val.jsonl
[Phase 1] train=50057  val=6256
[Phase 1] params=9,992,488
[Ep  1] train_ppl=823.6  val_ppl=250.3  val_emo=0.0001
  [ckpt] Saved → checkpoints/phase1/best_model.pt
[Ep  2] train_ppl=218.3  val_ppl=176.4  val_emo=0.0000
  [ckpt] Saved → checkpoints/phase1/best_model.pt
[Ep  3] train_ppl=172.3  val_ppl=152.8  val_emo=0.0000
  [ckpt] Saved → checkpoints/phase1/best_model.pt
[Ep  4] train_ppl=152.5  val_ppl=139.8  val_emo=0.0000
  [ckpt] Saved → checkpoints/phase1/best_model.pt
[Ep  5] train_ppl=140.5  val_ppl=132.4  val_emo=0.0000
  [ckpt] Saved → checkpoints/phase1/best_model.pt
[Ep  6] train_ppl=133.0  val_ppl=128.1  val_emo=0.0000
  [ckpt] Saved → checkpoints/phase1/best_model.pt
[Ep  7] train_ppl=128.1  val_ppl=125.1  val_emo=0.0000
  [ckpt] Saved → checkpoints/phase1/best_model.pt
[Ep

## Training Phase 2 (Optional)

After Phase 1, you can train on ESConv data for strategy-aware responses.

In [7]:
# Train Phase 2 (uncomment to run)
!python train_phase2.py --batch_size 8 --grad_accum 8 --epochs 10

[Phase 2] device=cuda
  [Dataset] Loaded 1669 samples from data/processed/esconv_train.jsonl
  [Dataset] Loaded 358 samples from data/processed/esconv_val.jsonl
[Phase 2] train=1669  val=358
[Phase 2] strategy weights: [0.22396807372570038, 1.40155029296875, 0.7732691168785095, 0.6499943137168884, 1.94998300075531, 1.2814173698425293, 0.8599086403846741, 0.8599086403846741]
[Phase 2] Loading Phase 1 ckpt: checkpoints/phase1/best_model.pt
  [ckpt] Loaded from checkpoints/phase1/best_model.pt  (epoch 10)
[Ep  1] ppl=2840.2  D1=0.007  D2=0.016  emo_f1=0.106  str_f1=0.054  tf=1.00
  [ckpt] Saved → checkpoints/phase2/best_model.pt
[Ep  2] ppl=1867.1  D1=0.008  D2=0.021  emo_f1=0.462  str_f1=0.340  tf=0.94
  [ckpt] Saved → checkpoints/phase2/best_model.pt
[Ep  3] ppl=1066.6  D1=0.004  D2=0.013  emo_f1=0.789  str_f1=0.308  tf=0.89
  patience 1/8  best_D2=0.021
[Ep  4] ppl=753.5  D1=0.004  D2=0.010  emo_f1=0.814  str_f1=0.247  tf=0.83
  patience 2/8  best_D2=0.021
[Ep  5] ppl=664.7  D1=0.005  

## Inference

Test the trained model.

In [10]:
# Run inference
!python inference.py --input "I feel really sad today." --checkpoint /content/MyChatbot/checkpoints/phase2/best_model.pt \
  --tokenizer_dir /content/MyChatbot/data/tokenizer

Loading model on cuda...
  [ckpt] Loaded from /content/MyChatbot/checkpoints/phase2/best_model.pt  (epoch 2)

[Emotion detected] caring
[Strategy selected] restatement
[Response] are you expensive about it?   I'm glad you about it.  People always you.  were a big point.   What should you do? What to get?  Ean?  are?  i feel?  Why are you feel good to you? Is it?  Do you get the on?  What did you?  Are you to ask?  What? Was it?  I hope you about that?  What happened?  What kind of the long?  It feels??  What a break?  are you just on??  What are you?  Thankfully?



## Download Checkpoints

Download the trained model checkpoints to your local machine.

In [ ]:
# Download checkpoints
from google.colab import files
import os

checkpoint_dir = 'checkpoints'
if os.path.exists(checkpoint_dir):
    for file in os.listdir(checkpoint_dir):
        if file.endswith('.pt'):
            files.download(os.path.join(checkpoint_dir, file))